# Gift-Eval Ablation Analysis

This notebook analyzes the effects of a TSFM's performance across different datasets.

**Ablation Levels:**
- Level 0: Original model (no ablation)
- Level 1: Ablate layers 10, heads-all
- Level 2: Ablate layers 10-11, heads-all
- Level 3: Ablate layers 10-11-12, heads-all
- Level 4: Ablate layers 10-11-12-13, heads-all
- Level 5: Ablate layers 10-11-12-13-14, heads-all
- Level 6: Ablate layers 10-11-12-13-14-15, heads-all


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
HOME_DIR = os.getenv("HOME")

In [ ]:
root_dir = os.path.join(HOME_DIR, "tsfm-lens")

In [ ]:
model_name = "TimesFM 2.5"

model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

In [ ]:
# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Analyzing metric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
save_figs = True
figs_save_dir = os.path.join(root_dir, "figs", f"gift-eval_{metric_pretty_name}", model_to_dir_mapping[model_name])
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
extra_description_str_by_key = {}
layers_srank = []
layers_srank_reverse = []

if model_name == "TimesFM 2.5":
    selected_layers_lst = [0, 5, 10, 19]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10, 12]
    n_divisor = 16  # number of heads in a layer
    extra_description_str_by_key = {
        "Layer 0": "srank",
        "Layer 5": "reverse srank",
        "Layer 10": "reverse srank",
        "Layer 19": "best random",
    }

    layers_srank = [0]
    layers_srank_reverse = [5, 10]

elif model_name == "Chronos Bolt":
    selected_layers_lst = [0, 2, 5, 11]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10]
    n_divisor = 12  # number of heads in a layer
    extra_description_str_by_key = {
        "Layer 0": "best random",
        "Layer 2": "reverse srank",
        "Layer 5": "best random",
        "Layer 11": "best random",
    }
    layers_srank = [0]
    layers_srank_reverse = [2]

elif model_name == "Toto":
    selected_layers_lst = [0, 2, 9, 11]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10]
    n_divisor = 12  # number of heads in a layer
    extra_description_str_by_key = {
        "Layer 0": "srank",
        "Layer 2": "best random",
        "Layer 9": "reverse srank",
        "Layer 11": "srank",
    }
    layers_srank = [0, 11]
    layers_srank_reverse = [9]
else:
    raise ValueError(f"Model {model_name} not supported yet")

# Build ablation files dictionary for all selected layers
ablated_files_dict = {}
ablation_meaning_str_dict = {}
ablation_level_meaning_str_dict = {}
ablation_level_meaning_alternative_str_dict = {}

ablation_meaning_str = "Ablation of Heads Across Layers"
ablation_level_meaning_str = "Pct of Heads Ablated per Layer"
ablation_level_meaning_alternative_str = "Number of Heads Ablated per Layer"

for selected_layers in selected_layers_lst:
    selected_layers_str = str(selected_layers)
    key = f"Layer {selected_layers_str}"
    if isinstance(selected_layers, list):
        key = f"Layers {'-'.join(str(layer) for layer in selected_layers)}"

    ablation_meaning_str_dict[key] = f"Ablation of Heads in Layer {selected_layers_str}"
    ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated"
    ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated"

    ablation_files = {
        0: "original_all_results.csv",
        **{n: f"head_layers_{selected_layers_str}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
        n_divisor: f"head_layers_{selected_layers_str}_heads-all_all_results.csv",
    }
    ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in ablation_files.items()}
    ablated_files_dict[key] = ablation_files

In [ ]:
ablation_files

In [ ]:
ablated_files_dict.keys()

In [ ]:
random_rseeds_lst = [
    123,
    99,
    42,
    688,
    22,
    33,
    56,
    78,
    357,
    234,
    567,
    890,
    1356,
    2233,
    3566,
    4891,
    222,
    333,
    444,
    555,
    777,
]
random_metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"rseed-{rseed}")
    for rseed in random_rseeds_lst
}

srank_rseeds_lst = [99, 42]
srank_metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"srank_rseed-{rseed}")
    for rseed in srank_rseeds_lst
}

srank_reverse_rseeds_lst = [99, 42]
srank_reverse_metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"srank_reverse_rseed-{rseed}")
    for rseed in srank_reverse_rseeds_lst
}

print(f"Random head selection metrics directory: {random_metrics_dir_dict}")
print(f"Stable rank (low to high) head selection metrics directory: {srank_metrics_dir_dict}")
print(f"Stable rank (high to low) head selection metrics directory: {srank_reverse_metrics_dir_dict}")


In [ ]:
layers_srank_str = [f"Layer {layer}" for layer in layers_srank]
layers_srank_reverse_str = [f"Layer {layer}" for layer in layers_srank_reverse]
print(f"Layers srank: {layers_srank_str}")
print(f"Layers srank reverse: {layers_srank_reverse_str}")

In [ ]:
# Load and combine CSV files across ablation levels and rseeds
# For each level, compute geometric mean across datasets for each rseed,
# then select the rseed with the lowest geometric mean
best_filepaths_by_key = {}
data_by_key = {}

for key, ablation_files in ablated_files_dict.items():
    data_by_key[key] = {}
    best_filepaths_by_key[key] = {}
    curr_data_frame = {}

    if key in layers_srank_str:
        metrics_dir_dict = srank_metrics_dir_dict
    elif key in layers_srank_reverse_str:
        metrics_dir_dict = srank_reverse_metrics_dir_dict
    else:
        metrics_dir_dict = random_metrics_dir_dict

    for level, filepath in ablation_files.items():
        rseed_data = {}
        available_rseeds = []

        # Load data for each rseed
        for rseed, metrics_dir in metrics_dir_dict.items():
            df_filepath = os.path.join(metrics_dir, filepath)
            if not os.path.exists(df_filepath):
                print(f"Level {level}, rseed {rseed}: Skipping (not found)")
                continue

            df = pd.read_csv(df_filepath)
            if len(df) != 97:
                print(f"Level {level}, rseed {rseed}: Skipping (incomplete)")
                continue

            print(f"Level {level}, rseed {rseed}: Loading")
            df["ablation_level"] = level
            rseed_data[rseed] = (df, df_filepath)
            available_rseeds.append(rseed)

        if not rseed_data:
            print(f"Level {level}: No data found, skipping")
            continue

        # Compute geometric mean for each rseed and select best
        rseed_gmeans = {}
        for rseed, (df, df_filepath) in rseed_data.items():
            selected_metric_gmean = gmean(df[SELECTED_METRIC])
            rseed_gmeans[rseed] = {
                f"gmean_{SELECTED_METRIC}": selected_metric_gmean,
                "filepath": df_filepath,
                "df": df,
            }

        # Select rseed with lowest geometric mean
        best_rseed = min(rseed_gmeans.keys(), key=lambda r: rseed_gmeans[r][f"gmean_{SELECTED_METRIC}"])
        best_df = rseed_gmeans[best_rseed]["df"]
        best_filepath = rseed_gmeans[best_rseed]["filepath"]

        best_filepaths_by_key[key][level] = {
            "best_rseed": best_rseed,
            "best_filepath": best_filepath,
            "available_rseeds": available_rseeds,
            f"gmean_{SELECTED_METRIC}": rseed_gmeans[best_rseed][f"gmean_{SELECTED_METRIC}"],
            "num_rseeds_evaluated": len(rseed_data),
        }

        curr_data_frame[level] = best_df
        print(
            f"Level {level}: Best rseed={best_rseed} (gmean={rseed_gmeans[best_rseed][f'gmean_{SELECTED_METRIC}']:.6f}) from {len(rseed_data)} rseeds"
        )

    all_data_for_key = pd.concat(curr_data_frame.values(), ignore_index=True)
    data_by_key[key] = all_data_for_key
    print(f"\nTotal: {len(all_data_for_key)} rows, {all_data_for_key['dataset'].nunique()} unique datasets")

In [ ]:
# Structure: best_filepaths_by_key[key][level] = {"rseed": ..., "filepath": ..., "overall_gmean": ..., "metric_gmeans": {...}}
print("Example of stored filepath information:")
print("=" * 80)
example_key = list(best_filepaths_by_key.keys())[1]
for key in [example_key]:  # Just show first key as example
    print(f"\nKey: {key}")
    for level in list(best_filepaths_by_key[key].keys()):
        info = best_filepaths_by_key[key][level]
        print(f"  Level {level}:")
        print(f"    Best rseed: {info['best_rseed']}, {info['best_filepath']}")
        print(f"    {SELECTED_METRIC} gmean: {info[f'gmean_{SELECTED_METRIC}']:.6f}")
        print(f"    {info['num_rseeds_evaluated']} rseeds evaluated: {info['available_rseeds']}")


In [ ]:
rng = np.random.default_rng(333)
layers_to_ablate = [10]
ablate_n_heads_per_layer = 10
heads_to_ablate = []
for layer in layers_to_ablate:
    heads_to_ablate_for_layer = rng.choice(list(range(12)), size=ablate_n_heads_per_layer, replace=False)
    heads_to_ablate.extend([(layer, head) for head in heads_to_ablate_for_layer])  # type: ignore
print(heads_to_ablate)

## Optional: Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE:
    for key in data_by_key.keys():
        all_data = data_by_key[key]
        print("\nLoading seasonal naive baseline...")
        seasonal_naive_path = os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv")
        seasonal_naive_df = pd.read_csv(seasonal_naive_path)
        print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets")

        print("\nNormalizing data by seasonal naive baseline...")
        all_data = normalize_by_seasonal_naive(all_data, seasonal_naive_df)
        data_by_key[key] = all_data
else:
    print("\nSkipping normalization.")

## Display Available Metrics


In [ ]:
# Display available metrics
metric_columns = [col for col in all_data.columns if "eval_metrics" in col]
print("Available metrics:")
for i, metric in enumerate(metric_columns, 1):
    print(f"{i}. {metric}")


## 1. Box Plot: Overall Performance Across Ablation Levels


In [ ]:
# Prepare data for box plot
box_data_by_key = {}
for key in data_by_key.keys():
    box_data = data_by_key[key][["dataset", "ablation_level", SELECTED_METRIC]].copy()
    # Remove any infinite or NaN values
    box_data = box_data.replace([np.inf, -np.inf], np.nan).dropna()
    box_data_by_key[key] = box_data


## 2. Line Plot: Median Performance Trend


In [ ]:
# Calculate median and percentiles for each ablation level and key
stats_by_level_by_key = {}
for key in data_by_key.keys():
    box_data = box_data_by_key[key]
    stats_by_level = (
        box_data.groupby("ablation_level")[SELECTED_METRIC]
        .agg(
            [
                "median",
                "mean",
                lambda x: gmean(x),
                lambda x: np.exp(np.std(np.log(x)) / np.sqrt(len(x))),  # geometric standard error (scale factor)
                lambda x: x.quantile(0.25),
                lambda x: x.quantile(0.75),
            ]
        )
        .rename(columns={"<lambda_0>": "geom_mean", "<lambda_1>": "geom_ste", "<lambda_2>": "q25", "<lambda_3>": "q75"})
        .reset_index()
    )
    stats_by_level_by_key[key] = stats_by_level

In [ ]:
single_layer_colors = plt.cm.get_cmap("cividis")(np.linspace(0.1, 0.95, len(data_by_key.keys())))
colors = list(single_layer_colors)

In [ ]:
data_by_key.keys()

In [ ]:
stats_by_level

In [ ]:
stats_by_level

In [ ]:
data_by_key.keys()

In [ ]:
extra_description_str_by_key.keys()

In [ ]:
# Create line plot with error bands
fig, ax = plt.subplots(figsize=(5, 4))

# Plot lines for each key
for i, key in enumerate(data_by_key.keys()):
    extra_description_str = extra_description_str_by_key[key]
    stats_by_level = stats_by_level_by_key[key]

    # Get baseline values

    if i == 0:
        baseline_geom_mean = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "geom_mean"].values[0]
        # Plot baseline reference lines
        ax.axhline(
            y=baseline_geom_mean,
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Baseline ({baseline_geom_mean:.3f})",
            alpha=0.7,
            zorder=4,
        )
        ax.text(
            0,
            baseline_geom_mean * 0.994,
            f"Baseline ({baseline_geom_mean:.3f})",
            color="red",
            fontsize=8,
            fontweight="bold",
            va="top",
            ha="left",
            zorder=100,
        )

        ax.axhline(y=1.01 * baseline_geom_mean, color="orangered", linestyle="--", linewidth=2, alpha=0.7)
        ax.text(
            0,
            (1.01 * baseline_geom_mean) * 1.004,
            "Baseline + 1%",
            color="orangered",
            fontsize=8,
            fontweight="bold",
            va="bottom",
            ha="left",
            zorder=100,
        )

    color = colors[i]
    # Plot geometric mean line with error envelope
    ax.plot(
        stats_by_level["ablation_level"] / n_divisor * 100,
        stats_by_level["geom_mean"],
        marker="^",
        markerfacecolor="None",
        markeredgecolor=color,
        markeredgewidth=2,
        linewidth=2.5,
        markersize=7,
        label=f"{key} ({extra_description_str})",
        color=color,
        linestyle="-",
        alpha=0.8,
    )

    ax.fill_between(
        stats_by_level["ablation_level"] / n_divisor * 100,
        stats_by_level["geom_mean"] * (stats_by_level["geom_ste"] ** 0.33),
        stats_by_level["geom_mean"] / (stats_by_level["geom_ste"] ** 0.33),
        alpha=0.05,
        color=color,
        zorder=-1,
    )

# Formatting
ax.set_xlabel(ablation_level_meaning_str + " (%)", fontweight="bold")
ax.set_ylabel(f"{metric_pretty_name} (Geom Mean)", fontweight="bold")
# ax.set_title(f"{ablation_meaning_str} ({model_name})", fontweight="bold", pad=20)
ax.set_title(model_name, fontweight="bold", pad=10)
ax.grid(True, alpha=0.2)
ax.legend(loc="upper left", frameon=True)

# Set x-axis ticks
all_levels = set()
for stats_by_level in stats_by_level_by_key.values():
    all_levels.update(stats_by_level["ablation_level"].unique())
xticks = sorted([x / n_divisor * 100 for x in all_levels])
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(x)}" if int(x) == x else f"{x:.1f}" for x in xticks], rotation=0)

# Set y-axis limits
ax.set_ylim(0.98 * baseline_geom_mean, 1.06 * baseline_geom_mean)

plt.tight_layout()
save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_line_plot_{model_name}.pdf")
if save_figs:
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved line plot to: {save_path}")
plt.show()

In [ ]:
# Calculate and display percentage change from baseline for each key
baseline_median = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "median"].values[0]
baseline_geom_mean = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "geom_mean"].values[0]

for key in data_by_key.keys():
    stats_by_level = stats_by_level_by_key[key]

    print(f"\nPercentage Change from Baseline (Level 0) for {key}:")
    print("=" * 70)
    for _, row in stats_by_level.iterrows():
        num_heads_ablated = int(row["ablation_level"])
        pct_heads_ablated = num_heads_ablated / n_divisor * 100

        median_change = ((row["median"] - baseline_median) / baseline_median) * 100
        curr_geom_mean = row["geom_mean"]
        geom_mean_change = ((curr_geom_mean - baseline_geom_mean) / baseline_geom_mean) * 100
        print(
            f"{num_heads_ablated} heads ({pct_heads_ablated:.1f}%) ablated: Median {median_change:+.2f}%, Geom Mean {geom_mean_change:+.2f}%"
        )
        print(f"==== Current Geom Mean: {curr_geom_mean:.3f}")


In [ ]:
all_data

## Identify Datasets close to Baseline

In [ ]:
# Get baseline values for each dataset
baseline_data = all_data[all_data["ablation_level"] == 0][["dataset", SELECTED_METRIC]].copy()
baseline_data.columns = ["dataset", "baseline_value"]

# Merge with all data to calculate relative performance
merged_data = all_data.merge(baseline_data, on="dataset")
# Calculate relative performance as difference from baseline (negative = improvement)
merged_data["relative_to_baseline"] = merged_data[SELECTED_METRIC] / merged_data["baseline_value"]

# Get max ablation level data
max_ablation_level = max(merged_data["ablation_level"].unique())
print(f"Max ablation level: {max_ablation_level}")
max_level_data = merged_data[merged_data["ablation_level"] == max_ablation_level]


# Helper function to build dataset info dictionary
def build_dataset_info(filtered_data):
    dataset_info = {}
    for _, row in filtered_data.iterrows():
        dataset = row["dataset"]
        dataset_data = merged_data[merged_data["dataset"] == dataset].sort_values("ablation_level")
        dataset_info[dataset] = {
            "baseline_value": row["baseline_value"],
            "max_ablation_value": row[SELECTED_METRIC],
            "relative_performance": row["relative_to_baseline"],
            "all_data": dataset_data,
        }
    return dataset_info


# Helper function to print dataset results
def print_dataset_results(title, datasets_dict, reverse_sort=False):
    sorted_datasets = sorted(datasets_dict.items(), key=lambda x: x[1]["relative_performance"], reverse=reverse_sort)
    print(f"{title}:")
    print("=" * 90)
    for i, (dataset, info) in enumerate(sorted_datasets[:10], 1):
        print(
            f"{i:2d}. {dataset:50s} | Relative: {info['relative_performance']:.4f} | "
            f"Baseline: {info['baseline_value']:.4f} | Max Ablation: {info['max_ablation_value']:.4f}"
        )


# Find and process resilient datasets (closest to zero - minimal change from baseline)
resilient_datasets = max_level_data[
    (max_level_data["relative_to_baseline"] <= 1.05) & (max_level_data["relative_to_baseline"] >= 0.95)
]
datasets_resilient = build_dataset_info(resilient_datasets)
print(f"Found {len(datasets_resilient)} most resilient datasets")
print_dataset_results("Most Resilient Datasets (closest to baseline at max ablation)", datasets_resilient)

improved_datasets = max_level_data[max_level_data["relative_to_baseline"] <= 1.0]
datasets_improved = build_dataset_info(improved_datasets)

print(f"Found {len(datasets_improved)} most improved datasets")
print_dataset_results("Most Improved Datasets (closest to baseline at max ablation)", datasets_improved)

# Find and process vulnerable datasets (most positive - worst degradation from baseline)
vulnerable_datasets = max_level_data[max_level_data["relative_to_baseline"] >= 1.0]
datasets_vulnerable = build_dataset_info(vulnerable_datasets)

print(f"\nFound {len(datasets_vulnerable)} least resilient datasets")
print_dataset_results(
    "Least Resilient Datasets (furthest from baseline at max ablation)", datasets_vulnerable, reverse_sort=True
)


In [ ]:
# Configuration
USE_MAX_IMPROVING_ABLATION = True  # True = highest ablation level with positive improvement
USE_BEST_ABLATION = True  # True = minimum metric value (ignored if USE_MAX_IMPROVING_ABLATION=True)
FIXED_ABLATION_LEVEL = 7  # Only used when both above are False
SHOW_ONLY_IMPROVEMENTS = True  # Filter to datasets with improvement over baseline
MAX_NUM_DATASETS_TO_PLOT = 12

mode_str = (
    "Max improving ablation"
    if USE_MAX_IMPROVING_ABLATION
    else "Best ablation"
    if USE_BEST_ABLATION
    else f"Fixed level {FIXED_ABLATION_LEVEL}"
)
print(f"Mode: {mode_str}")
print(f"Filter: {'Improvements only' if SHOW_ONLY_IMPROVEMENTS else 'All datasets'}")

# Pivot data: datasets × ablation levels
comparison_data = box_data.pivot(index="dataset", columns="ablation_level", values=SELECTED_METRIC)

# Prepare plotting data
plot_data = []
for dataset in comparison_data.index:
    baseline = comparison_data.loc[dataset, 0]
    ablation_cols = [c for c in comparison_data.columns if c != 0]

    if USE_MAX_IMPROVING_ABLATION:
        # Find highest ablation level with positive improvement
        improving_levels = [c for c in ablation_cols if comparison_data.loc[dataset, c] < baseline]
        best_level = max(improving_levels) if improving_levels else 0
        best_value = comparison_data.loc[dataset, best_level]
    elif USE_BEST_ABLATION:
        # Find best performing ablation (minimum value, excluding baseline)
        best_level = comparison_data.loc[dataset, ablation_cols].idxmin()
        best_value = comparison_data.loc[dataset, best_level]
    else:
        # Use fixed ablation level
        if FIXED_ABLATION_LEVEL not in comparison_data.columns:
            continue
        best_level = FIXED_ABLATION_LEVEL
        best_value = comparison_data.loc[dataset, FIXED_ABLATION_LEVEL]

    # Filter if needed
    if SHOW_ONLY_IMPROVEMENTS and best_value >= baseline:
        continue

    plot_data.append(
        {
            "dataset": dataset,
            "baseline": baseline,
            "ablation_value": best_value,
            "ablation_level": best_level,
            "improvement": baseline - best_value,
        }
    )

# Sort by improvement and limit to top N
plot_data.sort(key=lambda x: x["improvement"], reverse=True)
plot_data = plot_data[:MAX_NUM_DATASETS_TO_PLOT]

# Print statistics
print(f"\nTotal datasets: {len(comparison_data)}")
print(
    f"Datasets with improvement: {sum(1 for d in plot_data if d['improvement'] > 0)} "
    f"({sum(1 for d in plot_data if d['improvement'] > 0) / len(comparison_data) * 100:.1f}%)"
)
print(f"Displaying: {len(plot_data)}")
if plot_data:
    print(
        f"Improvement range: {min(d['improvement'] for d in plot_data):.4f} to "
        f"{max(d['improvement'] for d in plot_data):.4f}"
    )

# Plot
if not plot_data:
    print("\nNo datasets to display. Try SHOW_ONLY_IMPROVEMENTS = False")
else:
    fig, ax = plt.subplots(figsize=(6, max(6, len(plot_data) * 0.5)))
    y_pos = np.arange(len(plot_data))[::-1]
    bar_width = 0.4

    comparison_label = (
        "Max Ablation Level with Improvement"
        if USE_MAX_IMPROVING_ABLATION
        else "Best Ablation"
        if USE_BEST_ABLATION
        else f"Level {FIXED_ABLATION_LEVEL}"
    )

    # Plot bars
    ax.barh(
        y_pos - bar_width / 2,
        [d["baseline"] for d in plot_data],
        bar_width,
        label="Original",
        color="tab:green",
        alpha=0.6,
    )
    bars2 = ax.barh(
        y_pos + bar_width / 2,
        [d["ablation_value"] for d in plot_data],
        bar_width,
        label=comparison_label,
        color="tab:blue",
        alpha=0.6,
    )

    # Annotate with ablation levels
    for bar, d in zip(bars2, plot_data):
        ablation_level_percent = 100 * d["ablation_level"] / n_divisor
        ablation_level_percent_str = (
            int(ablation_level_percent)
            if int(ablation_level_percent) == ablation_level_percent
            else f"{ablation_level_percent:.1f}"
        )
        ax.text(
            bar.get_width(),
            bar.get_y() + bar.get_height() / 2,
            f" ({ablation_level_percent_str}%)",
            ha="left",
            va="center",
            fontsize=7,
            fontweight="bold",
        )

    # Format
    ax.set_yticks(y_pos)
    ax.set_yticklabels([d["dataset"] for d in plot_data], fontsize=8)
    ax.set_ylim(y_pos.min() - 0.5, y_pos.max() + 1.5)
    ax.set_xlabel(f"$\\mathbf{{{metric_pretty_name}}}$", fontsize=12, fontweight="bold")
    ax.set_ylabel("Dataset", fontsize=12, fontweight="bold")

    title = (
        f"{ablation_meaning_str} ({model_name})"
        if USE_MAX_IMPROVING_ABLATION
        else f"{ablation_meaning_str}: Best Ablations"
        if USE_BEST_ABLATION
        else f"{ablation_meaning_str}: Baseline vs Level {FIXED_ABLATION_LEVEL}"
    )
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=10, frameon=True)
    ax.grid(True, alpha=0.3, axis="x")

    plt.tight_layout()
    save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_datasets_improvement_bar_plot_{model_name}.pdf")
    if save_figs:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved heatmap to: {save_path}")
    plt.show()

    print(
        f"\n✓ Blue bars show {'HIGHEST ablation level with positive improvement' if USE_MAX_IMPROVING_ABLATION else 'ablation level with LOWEST metric (best performance)' if USE_BEST_ABLATION else f'performance at fixed level {FIXED_ABLATION_LEVEL}'}"
    )
    print(f"✓ Sorted by improvement (best at top)")
    if SHOW_ONLY_IMPROVEMENTS:
        print(f"✓ Filtered to improvements only")


In [ ]:
import seaborn as sns

# Create pivot table with only complete datasets (present in all ablation levels)
complete_datasets = box_data.groupby("dataset").size()
complete_datasets = complete_datasets[complete_datasets > 0].index

heatmap_data = box_data[box_data["dataset"].isin(complete_datasets)].pivot(
    index="dataset", columns="ablation_level", values=SELECTED_METRIC
)


# Custom sorting function for dataset names
def parse_dataset_name(name):
    """Parse dataset name into (dataset_name, freq, term) for sorting."""
    parts = name.split("/")
    if len(parts) != 3:
        return (name, "", "")

    dataset_name, freq, term = parts

    # Parse frequency for sorting
    freq_order = {"H": 1000, "D": 2000, "W": 3000, "M": 4000, "Q": 5000, "Y": 6000}
    freq_sort_key = int(freq[:-1]) if freq[-1] == "T" and freq[:-1].isdigit() else freq_order.get(freq, 10000)

    # Parse term for sorting
    term_order = {"short": 0, "medium": 1, "long": 2}
    term_sort_key = term_order.get(term, 3)

    return (dataset_name, freq_sort_key, term_sort_key)


# Sort and limit datasets
heatmap_data = heatmap_data.reindex(sorted(heatmap_data.index, key=parse_dataset_name))
if len(heatmap_data) > 100:
    print(f"Note: Showing top 50 datasets out of {len(heatmap_data)} total for readability")
    heatmap_data = heatmap_data.head(50)

# Split data into three equal parts
n_datasets = len(heatmap_data)
split_1, split_2 = n_datasets // 3, 2 * n_datasets // 3
data_parts = [heatmap_data.iloc[:split_1], heatmap_data.iloc[split_1:split_2], heatmap_data.iloc[split_2:]]

# Get global min/max for consistent colorbar
vmin, vmax = heatmap_data.min().min(), 3

# Create figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(20, max(8, split_1 * 0.3)))

# Plot heatmaps
for i, (ax, data) in enumerate(zip(axes, data_parts)):
    start_idx = i * split_1 + 1
    end_idx = start_idx + len(data) - 1

    sns.heatmap(data, annot=True, cmap="YlOrRd", vmin=vmin, vmax=vmax, cbar=False, linewidths=0.5, ax=ax)
    ax.set_xlabel(ablation_level_meaning_alternative_str, fontsize=12, fontweight="bold")
    ax.set_ylabel("Dataset" if i == 0 else "", fontsize=12, fontweight="bold")
    ax.set_title(f"Datasets {start_idx}-{end_idx}", fontsize=12, fontweight="bold")

# Add shared colorbar
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(f"$\\mathbf{{{metric_pretty_name}}}$", fontsize=12, fontweight="bold")

# Add overall title
fig.suptitle(f"{ablation_meaning_str} ({model_name})", fontsize=20, fontweight="bold", x=0.5, y=0.96)

plt.tight_layout(rect=[0, 0, 0.93, 0.96])
save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_datasets_heatmap_{model_name}.pdf")
if save_figs:
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved heatmap to: {save_path}")
plt.show()

print(f"\nShowing {len(heatmap_data)} datasets with complete data across all ablation levels (split into 3 columns)")


In [ ]:
len(box_data[SELECTED_METRIC])

In [ ]:
list(set(box_data["ablation_level"]))

In [ ]:
from scipy import stats

# Statistical tests
print("Computing Spearman correlation and Kruskal-Wallis test...")
print("... correlation between ablation level and metric values")
groups = [
    box_data[box_data["ablation_level"] == level][SELECTED_METRIC].values
    for level in sorted(box_data["ablation_level"].unique())
]
h_stat, p_value_kw = stats.kruskal(*groups)

correlation, p_value = stats.spearmanr(box_data["ablation_level"], box_data[SELECTED_METRIC])
geometric_means = box_data.groupby("ablation_level")[SELECTED_METRIC].apply(lambda x: np.prod(x) ** (1 / len(x)))
correlation_gmeans, p_value_gmeans = stats.spearmanr(list(set(box_data["ablation_level"])), list(geometric_means))
print("=" * 70)
print(
    f"Spearman Correlation: r={correlation:.4f}, p={p_value:.4e} "
    f"({'Significant' if p_value < 0.05 else 'Not significant'} "
    f"{'positive' if correlation > 0 else 'negative'})"
)

print(
    f"Spearman Correlation (geometric means): r={correlation_gmeans:.4f}, p={p_value_gmeans:.4e} "
    f"({'Significant' if p_value_gmeans < 0.05 else 'Not significant'} "
    f"{'positive' if correlation_gmeans > 0 else 'negative'})"
)

print(
    f"Kruskal-Wallis: H={h_stat:.4f}, p={p_value_kw:.4e} "
    f"({'Significant' if p_value_kw < 0.05 else 'Not significant'} difference)"
)

# Effect sizes relative to baseline
print(f"\nMedian Values and Effect Sizes:")
print("=" * 70)
baseline_median = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "median"].values[0]
for _, row in stats_by_level.iterrows():
    level, median = int(row["ablation_level"]), row["median"]
    print(f"Level {level}: {median:.4f} ({median / baseline_median:.4f}x baseline)")
